# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashiba713/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('/content/content_refresh_anonymized.csv')

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Unit of analysis + time window


* **Unit of analysis:** One row represents **one content item (`content_id`) with aggregated observed search and engagement measurements**.
* **Time window:** The starter release contains **rolling 30-day and 90-day observed metrics** (for example `impressions_last_30d`, `clicks_last_30d`, and `impressions_90d`). Because the public starter release is already aggregated, this notebook uses **the provided observed windows in the release** rather than filtering to a separate calendar month.


In [ ]:
# Verify the grain of the starter release
total_rows = len(df)
unique_content_ids = df['content_id'].nunique()

print('Total rows:', total_rows)
print('Unique content_id values:', unique_content_ids)
print('Grain valid:', total_rows == unique_content_ids)

## 2. Fields: feature / label / context / excluded

### Feature fields

These fields are used to create a **directional decision-support ranking** for content review prioritization.

* `impressions_last_30d`
* `clicks_last_30d`
* `sessions_last_30d`
* `ctr`
* `avg_position`
* `days_since_last_update`
* `content_age_days`
* `engagement_rate`
* `scroll_rate`

### Label field

The observed outcome field is:

* `trend_direction`

### Context fields

These fields provide identification or descriptive context but are not used as predictive features.

* `content_id`
* `client_id`
* `content_type`
* `main_intent`

### Excluded fields

These fields are deliberately excluded from the feature set:

* `trend_direction` — this is the observed outcome being ranked.
* `trend_pct` — this is label-derived directional information and would create **feature leakage**.


In [ ]:
# Define the field groups used in this notebook

feature_fields = [
    'impressions_last_30d',
    'clicks_last_30d',
    'sessions_last_30d',
    'ctr',
    'avg_position',
    'days_since_last_update',
    'content_age_days',
    'engagement_rate',
    'scroll_rate'
]

label_fields = ['trend_direction']

context_fields = [
    'content_id',
    'client_id',
    'content_type',
    'main_intent'
]

excluded_fields = ['trend_direction', 'trend_pct']

print('Feature fields:', feature_fields)
print('Label fields:', label_fields)
print('Context fields:', context_fields)
print('Excluded fields:', excluded_fields)

## 3. Verify it with queries (grain, counts, missing values, windows)

The queries below verify the grain, row counts, and feature availability of the starter release.

In [ ]:
# Query 1 — Grain check

total_rows = len(df)
unique_content_ids = df['content_id'].nunique()

print('--- Query 1: Grain check ---')
print('Total rows:', total_rows)
print('Unique content_id values:', unique_content_ids)
print('One row per content item:', total_rows == unique_content_ids)

In [ ]:
# Query 2 — Dataset size and observed windows

print('--- Query 2: Dataset size and observed windows ---')
print('Rows:', len(df))
print('Columns:', len(df.columns))

window_columns = [
    'impressions_last_30d',
    'clicks_last_30d',
    'sessions_last_30d',
    'impressions_prev_30d',
    'clicks_prev_30d',
    'sessions_prev_30d',
    'impressions_90d',
    'clicks_90d'
]

print('\\nObserved window columns:')
for col in window_columns:
    print('-', col)

In [ ]:
# Query 3 — Feature availability

availability_summary = df[feature_fields].notna().sum().to_frame('non_null_rows')
availability_summary['availability_pct'] = (
    availability_summary['non_null_rows'] / len(df) * 100
).round(2)

print('--- Query 3: Feature availability ---')
availability_summary

### Interpretation

* The grain check confirms that **each `content_id` appears once** in the starter release.
* The dataset contains **observed rolling 30-day, previous 30-day, and 90-day measurement windows**.
* The availability summary shows that the selected feature fields are **largely populated and suitable for directional decision-support analysis**.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

* This notebook uses the **public starter release**, which contains **aggregated observed search and engagement measurements** rather than raw daily Search Console or Analytics records.
* The analysis cannot identify **real client websites, URLs, or editorial intent** because the dataset is anonymized.
* The selected features support **directional decision-support for content refresh prioritization**; they do **not** predict Google’s ranking algorithm or guarantee future ranking outcomes.
* Because the release is already aggregated into **30-day, previous 30-day, and 90-day observed windows**, this notebook cannot measure **day-by-day causal effects** or isolate the exact reason why a page’s performance changed.


## Self-check

Before submitting, I confirmed that:

* [x] Every section above is filled with both **markdown reasoning and supporting code**
* [x] The notebook runs **top to bottom without errors**
* [x] No client names, URLs, or private search queries appear anywhere
* [x] The notebook uses careful FlyRank language: **observed, measured, directional, decision-support**
* [x] Label-derived fields (`trend_direction`, `trend_pct`) are **excluded from the feature set** to avoid feature leakage
